# ver3 — Full-data VAE + Latent Diffusion (latent dim=16)
이 노트북은 **전체 데이터(모든 스타일)**로 VAE(변이형 오토인코더; 확률적 압축/복원)를 학습하고,
잠재공간(latent space; 압축 공간)에서 Diffusion(확산모델; 노이즈→복원 반복)으로 (X,Y) 결합 데이터를 생성합니다.

이후 **Fidelity(원본 유사성)** + **Utility(유용성; 예측 성능 기여)**를 한 번에 평가합니다.

- 평가: KS-test(분포 유사도), PCD(상관구조 차이), DCR(최근접 거리)
- 예측: TRTR(원본 학습→원본 평가), TSTR(합성 학습→원본 평가), AUG(원본+합성 학습→원본 평가)


## 0. GPU 설정(필수)
- 이 셀을 **가장 먼저** 실행하세요.
- 이미 `torch`를 import한 상태면 커널 재시작 후 다시 실행하세요.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="2,3"


## 1. Imports(불러오기) & 설정(config; 하이퍼파라미터)


In [ ]:
from __future__ import annotations

import json, math, random
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, RandomSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.covariance import LedoitWolf
from sklearn.metrics import r2_score, mean_squared_error

from scipy.stats import ks_2samp

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

XLSX_PATH = Path("Supplemental Files and Figure source files.xlsx")
assert XLSX_PATH.exists(), f"엑셀 파일을 찾을 수 없습니다: {XLSX_PATH.resolve()}"

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path("runs") / f"ver3_full_{RUN_TAG}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LATENT_DIM = 16

# VAE
VAE_EPOCHS = 800
VAE_BATCH_SIZE = 512
VAE_LR = 1e-3
KL_BETA_MAX = 0.1
KL_WARMUP_EPOCHS = 200

# Diffusion
DIFF_T = 200
DIFF_EPOCHS = 3000
DIFF_BATCH_SIZE = 4096
DIFF_LR = 2e-4
STEPS_PER_EPOCH = 200

# Sampling
SYNTH_N = 10000  # 전체 합성 샘플 수(원본 175 train 기준 50~60배도 가능)
SAMPLE_BATCH = 4096

# Downstream predictor
DOWNSTREAM = "rf"  # rf or xgb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Visible GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))

## 2. 데이터 로드(load) & Train/Test split(층화)


In [ ]:
xls = pd.ExcelFile(XLSX_PATH)
df_x = xls.parse("Supplementary File S1")
df_y = xls.parse("Supplementary File S4")

meta_cols = ["beer", "beer_id", "tasting_category_fine"]
chem_cols = [c for c in df_x.columns if c not in meta_cols]
sens_cols = [c for c in df_y.columns if c not in meta_cols]

idx = np.arange(len(df_x))
style = df_x["tasting_category_fine"].astype(str)

idx_train, idx_test = train_test_split(
    idx,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
    stratify=style
)

df_train_x = df_x.iloc[idx_train].reset_index(drop=True)
df_test_x  = df_x.iloc[idx_test].reset_index(drop=True)
df_train_y = df_y.iloc[idx_train].reset_index(drop=True)
df_test_y  = df_y.iloc[idx_test].reset_index(drop=True)

df_train_joint = pd.concat([df_train_x[chem_cols], df_train_y[sens_cols]], axis=1)
df_test_joint  = pd.concat([df_test_x[chem_cols],  df_test_y[sens_cols]],  axis=1)

print("Train N:", len(df_train_joint), "Test N:", len(df_test_joint))
print("Dx:", len(chem_cols), "Dy:", len(sens_cols), "D:", df_train_joint.shape[1])

## 3. 전처리(StandardScaler; 표준화)


In [ ]:
scaler_joint = StandardScaler().fit(df_train_joint.values)
import joblib
joblib.dump(scaler_joint, OUT_DIR / "scaler_joint.joblib")

Z_train = scaler_joint.transform(df_train_joint.values).astype(np.float32)
Z_test  = scaler_joint.transform(df_test_joint.values).astype(np.float32)
input_dim = Z_train.shape[1]

## 4. PyTorch Dataset & DataLoader


In [ ]:
class ArrayDataset(Dataset):
    def __init__(self, arr: np.ndarray):
        self.arr = torch.from_numpy(arr).float()
    def __len__(self):
        return self.arr.shape[0]
    def __getitem__(self, idx):
        return self.arr[idx]

ds = ArrayDataset(Z_train)
val_size = max(1, int(0.1 * len(ds)))
train_size = len(ds) - val_size
ds_tr, ds_val = random_split(ds, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

sampler = RandomSampler(ds_tr, replacement=True, num_samples=STEPS_PER_EPOCH * VAE_BATCH_SIZE)
dl_tr = DataLoader(ds_tr, batch_size=VAE_BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
dl_val = DataLoader(ds_val, batch_size=min(VAE_BATCH_SIZE, len(ds_val)), shuffle=False, num_workers=2, pin_memory=True)

## 5. VAE 학습


In [ ]:
class MLPVAE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, hidden=(512, 256, 256)):
        super().__init__()
        h1, h2, h3 = hidden
        self.enc = nn.Sequential(
            nn.Linear(input_dim, h1), nn.SiLU(),
            nn.Linear(h1, h2), nn.SiLU(),
            nn.Linear(h2, h3), nn.SiLU(),
        )
        self.mu = nn.Linear(h3, latent_dim)
        self.logvar = nn.Linear(h3, latent_dim)
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, h3), nn.SiLU(),
            nn.Linear(h3, h2), nn.SiLU(),
            nn.Linear(h2, h1), nn.SiLU(),
            nn.Linear(h1, input_dim),
        )
    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.logvar(h)
    def reparam(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std
    def decode(self, z):
        return self.dec(z)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

def kl_beta(epoch: int, beta_max: float, warmup_epochs: int) -> float:
    if warmup_epochs <= 0:
        return beta_max
    return float(beta_max * min(1.0, epoch / warmup_epochs))

def vae_loss(x, recon, mu, logvar, beta: float):
    recon_loss = F.mse_loss(recon, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta*kl, recon_loss.detach(), kl.detach()

vae = MLPVAE(input_dim, LATENT_DIM).to(device)
if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    vae = nn.DataParallel(vae)
    print("Using DataParallel on", torch.cuda.device_count(), "GPUs")

opt = torch.optim.AdamW(vae.parameters(), lr=VAE_LR, weight_decay=1e-6)
scaler_amp = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

best_val = float("inf")
best_state = None
patience = 80
bad = 0
log = []

for epoch in range(1, VAE_EPOCHS+1):
    vae.train()
    beta = kl_beta(epoch, KL_BETA_MAX, KL_WARMUP_EPOCHS)
    tr_recon=[]
    tr_kl=[]
    for xb in dl_tr:
        xb = xb.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            recon, mu, logvar, _ = vae(xb)
            loss, rloss, kl = vae_loss(xb, recon, mu, logvar, beta)
        scaler_amp.scale(loss).backward()
        scaler_amp.step(opt)
        scaler_amp.update()
        tr_recon.append(rloss.item()); tr_kl.append(kl.item())

    # val
    vae.eval()
    with torch.no_grad():
        for xb in dl_val:
            xb = xb.to(device, non_blocking=True)
            recon, mu, logvar, _ = vae(xb)
            loss, rloss, kl = vae_loss(xb, recon, mu, logvar, KL_BETA_MAX)
            val_recon = rloss.item()
            val_kl = kl.item()
            break

    log.append({"epoch":epoch, "beta":beta,
                "train_recon":float(np.mean(tr_recon)),
                "train_kl":float(np.mean(tr_kl)),
                "val_recon":val_recon,
                "val_kl":val_kl})

    if epoch % 50 == 0 or epoch==1:
        print(f"[VAE] ep={epoch:4d} beta={beta:.3f} val_recon={val_recon:.4f} val_kl={val_kl:.4f}")

    if val_recon < best_val - 1e-6:
        best_val = val_recon
        bad = 0
        module = vae.module if isinstance(vae, nn.DataParallel) else vae
        best_state = {k: v.detach().cpu() for k,v in module.state_dict().items()}
    else:
        bad += 1
        if bad >= patience:
            print("Early stopping.")
            break

module = vae.module if isinstance(vae, nn.DataParallel) else vae
module.load_state_dict(best_state)

torch.save(best_state, OUT_DIR/"vae_state_dict.pt")
pd.DataFrame(log).to_csv(OUT_DIR/"vae_training_log.csv", index=False)
print("Best val recon:", best_val)

## 6. 잠재벡터 z 추출(encode)


In [ ]:
module.eval()
@torch.no_grad()
def encode_np(arr_scaled: np.ndarray) -> np.ndarray:
    x = torch.from_numpy(arr_scaled).float().to(device)
    mu, logvar = module.encode(x)
    return mu.detach().cpu().numpy()

z_train = encode_np(Z_train).astype(np.float32)
print("z_train:", z_train.shape)

ds_diff = ArrayDataset(z_train)
sampler_diff = RandomSampler(ds_diff, replacement=True, num_samples=STEPS_PER_EPOCH*DIFF_BATCH_SIZE)
dl_diff = DataLoader(ds_diff, batch_size=DIFF_BATCH_SIZE, sampler=sampler_diff, num_workers=4, pin_memory=True)

## 7. 잠재확산(Diffusion) 학습


In [ ]:
def linear_beta_schedule(T, beta_start=1e-4, beta_end=2e-2):
    return torch.linspace(beta_start, beta_end, T)

class SinusoidalTimeEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        device = t.device
        half = self.dim//2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=device) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0,1))
        return emb

class DiffusionMLP(nn.Module):
    def __init__(self, latent_dim, time_dim=128, hidden=512):
        super().__init__()
        self.time_emb = SinusoidalTimeEmb(time_dim)
        self.net = nn.Sequential(
            nn.Linear(latent_dim+time_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, latent_dim)
        )
    def forward(self, zt, t):
        te = self.time_emb(t)
        return self.net(torch.cat([zt, te], dim=1))

T = DIFF_T
betas = linear_beta_schedule(T).to(device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alphas_cumprod[:-1]], dim=0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
posterior_variance = betas*(1.0-alphas_cumprod_prev)/(1.0-alphas_cumprod)

def q_sample(z0, t, noise):
    a = sqrt_alphas_cumprod[t].unsqueeze(1)
    b = sqrt_one_minus_alphas_cumprod[t].unsqueeze(1)
    return a*z0 + b*noise

diff = DiffusionMLP(LATENT_DIM).to(device)
if torch.cuda.is_available() and torch.cuda.device_count()>=2:
    diff = nn.DataParallel(diff)

opt = torch.optim.AdamW(diff.parameters(), lr=DIFF_LR, weight_decay=1e-6)
scaler_amp = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

log=[]
diff.train()
for epoch in range(1, DIFF_EPOCHS+1):
    losses=[]
    for z0 in dl_diff:
        z0 = z0.to(device, non_blocking=True)
        bsz = z0.size(0)
        t = torch.randint(0, T, (bsz,), device=device).long()
        noise = torch.randn_like(z0)
        zt = q_sample(z0, t, noise)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            eps = diff(zt, t)
            loss = F.mse_loss(eps, noise)
        scaler_amp.scale(loss).backward()
        scaler_amp.step(opt)
        scaler_amp.update()
        losses.append(loss.item())
    m=float(np.mean(losses)); log.append({"epoch":epoch, "loss":m})
    if epoch%200==0 or epoch==1:
        print(f"[DIFF] ep={epoch:4d} loss={m:.6f}")

module_d = diff.module if isinstance(diff, nn.DataParallel) else diff
torch.save(module_d.state_dict(), OUT_DIR/"diff_state_dict.pt")
pd.DataFrame(log).to_csv(OUT_DIR/"diff_training_log.csv", index=False)

## 8. 전체 합성 데이터 생성(sampling)


In [ ]:
module_d = diff.module if isinstance(diff, nn.DataParallel) else diff
module_d.eval()
module.eval()

@torch.no_grad()
def p_sample_step(zt, t):
    eps = module_d(zt, t)
    beta_t = betas[t].unsqueeze(1)
    alpha_t = alphas[t].unsqueeze(1)
    a_bar = alphas_cumprod[t].unsqueeze(1)
    mean = (1/torch.sqrt(alpha_t))*(zt - (beta_t/torch.sqrt(1-a_bar))*eps)
    if t[0].item()==0:
        return mean
    var = posterior_variance[t].unsqueeze(1)
    return mean + torch.sqrt(var)*torch.randn_like(zt)

@torch.no_grad()
def sample_latents(n:int, batch:int):
    out=[]; done=0
    while done<n:
        cur=min(batch, n-done)
        zt=torch.randn(cur, LATENT_DIM, device=device)
        for step in reversed(range(T)):
            t = torch.full((cur,), step, device=device, dtype=torch.long)
            zt = p_sample_step(zt, t)
        out.append(zt.detach().cpu().numpy()); done += cur
    return np.concatenate(out, axis=0)

z_synth = sample_latents(SYNTH_N, SAMPLE_BATCH).astype(np.float32)
print("z_synth:", z_synth.shape)

@torch.no_grad()
def decode_latents(z_np: np.ndarray) -> np.ndarray:
    z = torch.from_numpy(z_np).float().to(device)
    xhat = module.decode(z)
    return xhat.detach().cpu().numpy()

joint_synth_scaled = decode_latents(z_synth).astype(np.float32)
joint_synth = scaler_joint.inverse_transform(joint_synth_scaled).astype(np.float32)

Xs = joint_synth[:, :len(chem_cols)]
Ys = joint_synth[:, len(chem_cols):]
df_synth_x = pd.DataFrame(Xs, columns=chem_cols)
df_synth_y = pd.DataFrame(Ys, columns=sens_cols)

df_synth_x.to_csv(OUT_DIR/"synth_X.csv", index=False)
df_synth_y.to_csv(OUT_DIR/"synth_Y.csv", index=False)

## 9. Fidelity 평가(KS, PCD X–X, PCD X–Y, DCR)


In [ ]:
def ks_summary(real: pd.DataFrame, synth: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for c in real.columns:
        r=real[c].values; s=synth[c].values
        r=r[np.isfinite(r)]; s=s[np.isfinite(s)]
        if len(r)<2 or len(s)<2: 
            continue
        stat,p=ks_2samp(r,s)
        rows.append({"col":c, "ks":float(stat), "p":float(p)})
    return pd.DataFrame(rows).sort_values("ks", ascending=False)

def corr_shrinkage(arr: np.ndarray) -> np.ndarray:
    lw = LedoitWolf().fit(arr)
    cov = lw.covariance_
    d = np.sqrt(np.diag(cov))
    corr = cov/(d[:,None]*d[None,:]+1e-12)
    return np.clip(corr, -1, 1)

def pcd_xx(real_x: np.ndarray, synth_x: np.ndarray) -> float:
    c1=corr_shrinkage(real_x); c2=corr_shrinkage(synth_x)
    return float(np.linalg.norm(c1-c2, ord="fro")/c1.size)

def pcd_xy(real_joint: np.ndarray, synth_joint: np.ndarray, dx:int) -> float:
    c1=corr_shrinkage(real_joint); c2=corr_shrinkage(synth_joint)
    b1=c1[:dx, dx:]; b2=c2[:dx, dx:]
    return float(np.linalg.norm(b1-b2, ord="fro")/b1.size)

def dcr(real_scaled: np.ndarray, synth_scaled: np.ndarray) -> np.ndarray:
    nn = NearestNeighbors(n_neighbors=1).fit(real_scaled)
    dist,_ = nn.kneighbors(synth_scaled)
    return dist.ravel()

real_joint = df_train_joint
synth_joint = pd.concat([df_synth_x, df_synth_y], axis=1)

ks_df = ks_summary(real_joint, synth_joint)
ks_df.to_csv(OUT_DIR/"fidelity_ks.csv", index=False)

pxx = pcd_xx(df_train_x[chem_cols].values.astype(np.float32), df_synth_x.values.astype(np.float32))
pxy = pcd_xy(df_train_joint.values.astype(np.float32), synth_joint.values.astype(np.float32), dx=len(chem_cols))

# DCR in scaled space
synth_scaled = scaler_joint.transform(synth_joint.values).astype(np.float32)
dcr_vals = dcr(Z_train, synth_scaled)

fidelity = {
    "pcd_xx": pxx,
    "pcd_xy": pxy,
    "dcr_mean": float(dcr_vals.mean()),
    "dcr_median": float(np.median(dcr_vals)),
    "dcr_p05": float(np.quantile(dcr_vals, 0.05)),
    "dcr_p95": float(np.quantile(dcr_vals, 0.95)),
    "ks_mean": float(ks_df["ks"].mean()),
    "ks_median": float(ks_df["ks"].median()),
}
with open(OUT_DIR/"fidelity_summary.json","w",encoding="utf-8") as f:
    json.dump(fidelity, f, ensure_ascii=False, indent=2)

print("Fidelity summary:", fidelity)
display(ks_df.head(10))

## 10. Utility 평가(TRTR, TSTR, AUG)


In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def eval_multi(y_true, y_pred):
    r2 = float(r2_score(y_true, y_pred, multioutput="uniform_average"))
    return r2, rmse(y_true, y_pred)

def make_model(kind: str):
    kind=kind.lower()
    if kind=="rf":
        from sklearn.ensemble import RandomForestRegressor
        return RandomForestRegressor(
            n_estimators=500, random_state=SEED, n_jobs=-1, max_features="sqrt"
        )
    elif kind=="xgb":
        from xgboost import XGBRegressor
        from sklearn.multioutput import MultiOutputRegressor
        base = XGBRegressor(
            n_estimators=1500,
            learning_rate=0.03,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
        )
        return MultiOutputRegressor(base, n_jobs=-1)
    else:
        raise ValueError

# real arrays
Xtr = df_train_x[chem_cols].values.astype(np.float32)
Ytr = df_train_y[sens_cols].values.astype(np.float32)
Xte = df_test_x[chem_cols].values.astype(np.float32)
Yte = df_test_y[sens_cols].values.astype(np.float32)

# synth arrays
Xs = df_synth_x.values.astype(np.float32)
Ys = df_synth_y.values.astype(np.float32)

# downstream scaling (fit on real train)
sx = StandardScaler().fit(Xtr)
Xtr_s = sx.transform(Xtr)
Xte_s = sx.transform(Xte)
Xs_s  = sx.transform(Xs)

results=[]

# TRTR
m = make_model(DOWNSTREAM)
m.fit(Xtr_s, Ytr)
pred = m.predict(Xte_s)
r2_trtr, rm_trtr = eval_multi(Yte, pred)
results.append({"setting":"TRTR", "r2":r2_trtr, "rmse":rm_trtr})
print("[TRTR] R2", r2_trtr, "RMSE", rm_trtr)

# TSTR (train synth only)
m = make_model(DOWNSTREAM)
m.fit(Xs_s, Ys)
pred = m.predict(Xte_s)
r2_tstr, rm_tstr = eval_multi(Yte, pred)
results.append({"setting":"TSTR", "r2":r2_tstr, "rmse":rm_tstr})
print("[TSTR] R2", r2_tstr, "RMSE", rm_tstr)

# AUG (real + synth)
# use n_aug = len(train) (1x) by default
n_aug = min(len(Xs_s), len(Xtr_s))
idx_aug = np.random.RandomState(SEED).choice(len(Xs_s), size=n_aug, replace=False)
X_aug = np.vstack([Xtr_s, Xs_s[idx_aug]])
Y_aug = np.vstack([Ytr, Ys[idx_aug]])

m = make_model(DOWNSTREAM)
m.fit(X_aug, Y_aug)
pred = m.predict(Xte_s)
r2_aug, rm_aug = eval_multi(Yte, pred)
results.append({"setting":"AUG", "r2":r2_aug, "rmse":rm_aug,
                "delta_r2_vs_TRTR": r2_aug - r2_trtr,
                "delta_rmse_vs_TRTR": rm_aug - rm_trtr})
print("[AUG] R2", r2_aug, "RMSE", rm_aug, "ΔR2", r2_aug-r2_trtr)

df_res = pd.DataFrame(results)
df_res.to_csv(OUT_DIR/"utility_results.csv", index=False)
display(df_res)

## 11. 스타일별(style-wise) 성능 분해(선택)
전체 평균 성능만 보면 특정 스타일에서만 좋아졌는지 놓칠 수 있습니다.
TRTR vs AUG를 스타일별로 분해해서 확인합니다.


In [ ]:
# Re-fit TRTR and AUG models to reuse predictions for style breakdown
def fit_and_predict(model, Xtr, Ytr, Xte):
    model.fit(Xtr, Ytr)
    return model.predict(Xte)

# TRTR pred already computed above? re-do for clarity
m_trtr = make_model(DOWNSTREAM)
pred_trtr = fit_and_predict(m_trtr, Xtr_s, Ytr, Xte_s)

n_aug = min(len(Xs_s), len(Xtr_s))
idx_aug = np.random.RandomState(SEED).choice(len(Xs_s), size=n_aug, replace=False)
X_aug = np.vstack([Xtr_s, Xs_s[idx_aug]])
Y_aug = np.vstack([Ytr, Ys[idx_aug]])
m_aug = make_model(DOWNSTREAM)
pred_aug = fit_and_predict(m_aug, X_aug, Y_aug, Xte_s)

styles_test = df_test_x["tasting_category_fine"].astype(str).values
unique_styles = sorted(pd.unique(styles_test))

rows=[]
for st in unique_styles:
    mask = (styles_test == st)
    if mask.sum() < 3:
        continue
    r2_t, rm_t = eval_multi(Yte[mask], pred_trtr[mask])
    r2_a, rm_a = eval_multi(Yte[mask], pred_aug[mask])
    rows.append({
        "style": st,
        "n_test": int(mask.sum()),
        "r2_trtr": r2_t,
        "r2_aug": r2_a,
        "delta_r2": r2_a - r2_t,
        "rmse_trtr": rm_t,
        "rmse_aug": rm_a,
        "delta_rmse": rm_a - rm_t,
    })

df_style = pd.DataFrame(rows).sort_values("delta_r2", ascending=False)
df_style.to_csv(OUT_DIR/"utility_style_breakdown.csv", index=False)
display(df_style.head(12))

## 12. 저장된 결과 파일
- `vae_state_dict.pt`, `diff_state_dict.pt`
- `synth_X.csv`, `synth_Y.csv`
- `fidelity_*`, `utility_*`


In [ ]:
config = {
    "LATENT_DIM": LATENT_DIM,
    "VAE_EPOCHS": VAE_EPOCHS,
    "DIFF_T": DIFF_T,
    "DIFF_EPOCHS": DIFF_EPOCHS,
    "SYNTH_N": SYNTH_N,
    "DOWNSTREAM": DOWNSTREAM,
    "SEED": SEED,
}
with open(OUT_DIR/"config.json","w",encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print("Done. Outputs saved to:", OUT_DIR.resolve())